# Notebook 03: Classical Machine Learning Baselines (CPU)

**Objective**: Benchmark non-neural classical classifiers (Random Forest, Logistic Regression, Linear SVM) on statistical time-series summary features.

Features extracted per window ($133 \times 4 = 532$ features):
- Mean
- Standard Deviation
- Minimum
- Maximum

**Compute Environment**: CPU Safe.

## 0. Environment Setup & Repository Bootstrap

In [ ]:
import sys
import os
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/Opportunity_HAR")
    if not REPO_DIR.exists():
        print("Cloning Opportunity_HAR from GitHub...")
        !git clone https://github.com/DhruvikaRajput/Opportunity_HAR.git {REPO_DIR}
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)
    !pip install -q -r requirements.txt
else:
    current = Path.cwd()
    REPO_DIR = current.parent if current.name == "notebooks" else current
    if str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    os.chdir(REPO_DIR)

import src.api as api
env_info = api.setup_project()
print(f"Project Initialized on: {env_info['device_name']} (CUDA: {env_info['cuda_available']})")

## 1. Configure Experiment Mode

In [ ]:
# Choose execution mode: 'DEBUG' (tiny sample), 'FAST' (moderate estimators), or 'FULL' (full ensemble)
EXPERIMENT_MODE = "FAST"
print(f"Experiment Mode: {EXPERIMENT_MODE}")

## 2. Load Processed Windows & Extract Statistical Features

In [ ]:
from src.models.classical import extract_statistical_features

try:
    subset = api.load_preprocessed_subset("opportunity_locomotion_subset.npz")
    X_train, y_train = subset["X_train"], subset["y_train"]
    X_test, y_test = subset["X_test"], subset["y_test"]

    if EXPERIMENT_MODE == "DEBUG":
        X_train, y_train = X_train[:200], y_train[:200]
        X_test, y_test = X_test[:100], y_test[:100]

    print("Extracting 532 statistical features per window...")
    feats_train = extract_statistical_features(X_train)
    feats_test = extract_statistical_features(X_test)

    print(f"Training Features Shape: {feats_train.shape}")
    print(f"Test Features Shape    : {feats_test.shape}")
except FileNotFoundError:
    print("[NOTICE] Preprocessed data not found. Run Notebook 02 first to preprocess the dataset.")

## 3. Train & Evaluate Classical Models on CPU

In [ ]:
from src.models.classical import train_random_forest, train_logistic_regression, train_linear_svm
import pandas as pd

n_trees = 20 if EXPERIMENT_MODE in ["DEBUG", "FAST"] else 100

print(f"Training Random Forest (n_estimators={n_trees})...")
rf_model, rf_metrics, rf_cm = train_random_forest(feats_train, y_train, feats_test, y_test, n_estimators=n_trees)

print("Training Logistic Regression...")
lr_model, lr_metrics, lr_cm = train_logistic_regression(feats_train, y_train, feats_test, y_test)

print("Training Linear SVM...")
svm_model, svm_metrics, svm_cm = train_linear_svm(feats_train, y_train, feats_test, y_test)

results_df = pd.DataFrame([
    {"Model": "Random Forest", "Accuracy": rf_metrics["accuracy"], "Macro F1": rf_metrics["macro_f1"], "Weighted F1": rf_metrics["weighted_f1"]},
    {"Model": "Logistic Regression", "Accuracy": lr_metrics["accuracy"], "Macro F1": lr_metrics["macro_f1"], "Weighted F1": lr_metrics["weighted_f1"]},
    {"Model": "Linear SVM", "Accuracy": svm_metrics["accuracy"], "Macro F1": svm_metrics["macro_f1"], "Weighted F1": svm_metrics["weighted_f1"]},
])

print("\n" + "=" * 55)
print(" CLASSICAL BASELINES SUMMARY")
print("=" * 55)
display_df = results_df.copy()
for col in ["Accuracy", "Macro F1", "Weighted F1"]:
    display_df[col] = display_df[col].apply(lambda x: f"{x*100:.2f}%")
print(display_df.to_string(index=False))
print("=" * 55)

## 4. Visualize Confusion Matrix (Random Forest)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))
sns.heatmap(rf_cm, annot=True, fmt="d", cmap="Blues", xticklabels=subset["class_names"], yticklabels=subset["class_names"])
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted Activity")
plt.ylabel("True Activity")
plt.tight_layout()
plt.show()